# 12 — Coarser grain: does aggregation reveal structure?

The provider-grain forecasting question is closed (07–09). The one untested direction: aggregate the same C_110 series to **ICB level** and to **England level** and re-ask whether anything beats persistence at 1/3/6/12 months. Aggregation cancels provider-level noise, so if seasonality or drift exists at system level, this is where it must surface.

Two protocol notes:

- **Constant membership.** Aggregating a churning provider set fabricates level jumps at every entry/exit, so the aggregate uses only providers present in all 96 months (declared composition rule, not tuned — the count is printed).
- Everything else unchanged: splits by target month, validation targets in FY 2024-25, sealed year unreachable (asserted), persistence in every table, both declared regimes for drift.

In [1]:
import pandas as pd                          # tables
import numpy as np                           # array arithmetic
from pathlib import Path

PROC = Path("../data/processed")
TEST_START = pd.Timestamp("2025-04-01")      # sealed test year - no target may reach it
VAL_START = pd.Timestamp("2024-04-01")       # validation targets: FY 2024-25
COVID = pd.Timestamp("2020-03-01")           # declared regime break

panel = pd.read_csv(PROC / "rtt_kh03_canonical_panel.csv", parse_dates=["month"])
dev = panel[panel["month"] < TEST_START]

# the usual C_110 stable public cohort, then constant membership: present all 96 months,
# so the aggregate series never jumps because a provider entered or left
df = dev[(dev["tfunc_code"] == "C_110") & dev["fl_client"] & dev["is_public"]
         & (dev["n_months_active"] == 96)].copy()
print(df["provider_code"].nunique(), "constant-membership providers (of the 130-ish stable cohort)")

icb = (df[df["icb_code"].notna()].groupby(["icb_code", "month"], as_index=False)["total_waiting"]
         .sum().rename(columns={"icb_code": "series"}))
nat = df.groupby("month", as_index=False)["total_waiting"].sum().assign(series="ENGLAND")
s = pd.concat([icb, nat], ignore_index=True).sort_values(["series", "month"]).reset_index(drop=True)
print(s["series"].nunique() - 1, "ICB series + ENGLAND;",
      int(s.groupby("series")["month"].count().min()), "months in the shortest series")

115 constant-membership providers (of the 130-ish stable cohort)
35 ICB series + ENGLAND; 84 months in the shortest series


## All horizons, both grains

Per horizon and grain: persistence, same-month-last-year (skipped at *h* = 12 where it *is* persistence), and drift (training-mean *h*-month change) in both regimes. MAE scales differ between grains (an ICB row aggregates several providers; England aggregates all) — each grain is judged only against its own persistence.

In [2]:
# Goal: one results table - MAE/RMSE per (horizon, grain, model, regime) on FY 2024-25 targets.
g = s.groupby("series")["total_waiting"]
results = []

def rec(h, grain, model, regime, err):
    results.append({"h": h, "grain": grain, "model": model, "regime": regime,
                    "mae": np.abs(err).mean(), "rmse": np.sqrt((err ** 2).mean()), "n": len(err)})

for h in (1, 3, 6, 12):
    d = s.copy()
    d["target"] = g.shift(-h)                # the series level h months later
    d["target_month"] = d["month"] + pd.DateOffset(months=h)
    d["seasonal"] = g.shift(12 - h)          # the actual level 12 months before the target
    d = d.dropna(subset=["target"])
    assert d["target_month"].max() < TEST_START   # a sealed-year target would trip this

    for grain, dd in [("icb", d[d["series"] != "ENGLAND"]), ("national", d[d["series"] == "ENGLAND"])]:
        tr = dd[dd["target_month"] < VAL_START]
        va = dd[dd["target_month"] >= VAL_START]
        truth = va["target"].to_numpy()

        rec(h, grain, "persistence", "-", truth - va["total_waiting"].to_numpy())
        if h != 12:                          # at h=12 same-month-last-year IS persistence
            rec(h, grain, "seasonal", "-", truth - va["seasonal"].to_numpy())
        for reg, t in [("full", tr), ("post", tr[tr["month"] >= COVID])]:
            drift = (t["target"] - t["total_waiting"]).mean()
            rec(h, grain, "drift", reg, truth - (va["total_waiting"] + drift).to_numpy())

res = pd.DataFrame(results)
for grain in ("icb", "national"):
    print(f"===== {grain} =====")
    for h in (1, 3, 6, 12):
        print(f"--- {h}-month ahead ---")
        for _, r in res[(res["h"] == h) & (res["grain"] == grain)].sort_values("mae").iterrows():
            print(f"  {r['model']:12s} {r['regime']:5s} MAE {r['mae']:9,.0f}   RMSE {r['rmse']:9,.0f}   (n={r['n']})")

===== icb =====
--- 1-month ahead ---
  persistence  -     MAE       310   RMSE       675   (n=420)
  drift        full  MAE       329   RMSE       679   (n=420)
  drift        post  MAE       341   RMSE       684   (n=420)
  seasonal     -     MAE     1,523   RMSE     2,205   (n=420)
--- 3-month ahead ---
  persistence  -     MAE       657   RMSE     1,151   (n=420)
  drift        full  MAE       738   RMSE     1,182   (n=420)
  drift        post  MAE       798   RMSE     1,216   (n=420)
  seasonal     -     MAE     1,523   RMSE     2,205   (n=420)
--- 6-month ahead ---
  persistence  -     MAE     1,077   RMSE     1,712   (n=420)
  drift        full  MAE     1,279   RMSE     1,833   (n=420)
  drift        post  MAE     1,424   RMSE     1,939   (n=420)
  seasonal     -     MAE     1,523   RMSE     2,205   (n=420)
--- 12-month ahead ---
  persistence  -     MAE     1,523   RMSE     2,205   (n=420)
  drift        full  MAE     1,992   RMSE     2,596   (n=420)
  drift        post  MAE   

In [3]:
# Goal: store the results table so these numbers stay re-readable without a rerun.
res.to_csv(PROC / "val_horizons_coarse_c110.csv", index=False)
print(res.shape, "->", PROC / "val_horizons_coarse_c110.csv")

(30, 7) -> ..\data\processed\val_horizons_coarse_c110.csv
